# 逐次ベイズ最適化：初期観測から評価予算終了まで

このNotebookは、**1つのベイズ最適化キャンペーンを最初から最後まで運用する基準例**です。初期観測 → GPの `fit()` → 事後予測 → 獲得関数 → 候補提案 → 実験 → データ追加 → 再学習を繰り返します。

1次元・単一目的の最大化問題を使い、**評価予算、観測済み最良値（incumbent）、実験履歴、終了条件**まで確認します。既存の `SingleTaskGP` と `optimize_acqf` を利用し、独自のBOフレームワークは定義しません。


## このNotebookで学ぶBOの全体像

1. **初期実験**：まだGPが学習できない段階で、複数の入力を測定する
2. **GP学習**：観測済みの `(X,Y)` から予測分布を作る
3. **獲得関数**：良い目的値を見つける価値を評価する
4. **候補最適化**：次の入力を1点提案する
5. **実験と更新**：観測結果を追加し、残り予算があれば繰り返す

`q=1` の逐次BOでは、**新しい観測が得られるたびにGPを更新**します。`q=2` の同時実験や非同期BOとは異なります。

## 1. Imports and campaign configuration

In [ ]:
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import SingleTaskGP
from robotorchan.optim import optimize_acqf

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

N_INITIAL = 6
N_BO_STEPS = 5
bounds = torch.tensor([[0.0], [1.0]])

### 評価予算の考え方

`N_INITIAL=6` は初期観測数、`N_BO_STEPS=5` は**最初の1回の候補提案・評価の後に追加する反復回数**です。

したがって最終評価回数は `6+1+5=12` 回です。初期観測を含む予算と、BO反復に割り当てる予算を混同しないようにします。

## 2. Expensive black-box objective

実務ではこの関数呼び出しが実験、シミュレーション、製造評価などに置き換わります。BO loopはこの評価責務をsurrogateやacquisitionへ隠しません。

In [ ]:
def expensive_objective(X: torch.Tensor) -> torch.Tensor:
    return (
        0.75 * torch.sin(3.0 * torch.pi * X)
        + 0.30 * torch.cos(7.0 * torch.pi * X)
        + 0.35 * X
    )


initial_X = torch.linspace(0.05, 0.95, N_INITIAL).unsqueeze(-1)
initial_Y = expensive_objective(initial_X)
train_X = initial_X.clone()
train_Y = initial_Y.clone()

print("initial evaluations:", train_X.shape[0])
print("initial best:", train_Y.max().item())

### 初期データと真の目的関数

`expensive_objective()` は実務では試験装置・シミュレータ・製造工程の測定に相当します。GPがこの式を直接知っているわけではありません。

初期データの6点を測定してから学習を始めます。最良観測値は、GPの予測最大値ではなく**実際に測定したYの最大値**です。

## 3. One complete BO decision

各decisionではmodelを現在のcompleted observationsから作り直します。

### 最初のGP学習と候補提案

`SingleTaskGP.fit()` でGPを学習し、`model.posterior(grid)` で予測平均と不確実性を取得します。

`qLogExpectedImprovement` は観測済み最良値 `best_f` に対する改善を評価します。`optimize_acqf(q=1)` は獲得関数を最大化する**次の実験入力1点**を返します。

`acq_value` は獲得関数値であり、実際の測定結果 `observed_Y` ではありません。

In [ ]:
model = SingleTaskGP(train_X, train_Y)
model.fit()
model.eval()

grid = torch.linspace(0.0, 1.0, 101).unsqueeze(-1)
posterior = model.posterior(grid)
print("posterior mean shape:", posterior.mean.shape)

sampler = SobolQMCNormalSampler(torch.Size([128]), seed=101)
acquisition = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
    sampler=sampler,
)
candidate, acq_value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
)
candidate = candidate.detach()
observed_Y = expensive_objective(candidate)

print("candidate:", candidate.item())
print("acquisition value:", acq_value.item())
print("observed value:", observed_Y.item())

### 最初の候補を評価する前後のGP予測

新しい候補を測定する前のGP予測を可視化します。測定した点は図に表示しますが、**次のセルでデータに追加するまでは学習済みGPに反映されません**。

In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():
    initial_posterior = model.posterior(grid)
    initial_mean = initial_posterior.mean.squeeze(-1)
    initial_std = initial_posterior.variance.clamp_min(0).sqrt().squeeze(-1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(grid.squeeze(-1), expensive_objective(grid).squeeze(-1), label="True objective")
ax.plot(grid.squeeze(-1), initial_mean, label="GP mean")
ax.fill_between(grid.squeeze(-1), initial_mean - 2*initial_std, initial_mean + 2*initial_std, alpha=0.2, label="Mean ± 2 SD")
ax.scatter(initial_X.squeeze(-1), initial_Y.squeeze(-1), s=30, label="Initial observations")
ax.scatter(candidate.squeeze(-1), observed_Y.squeeze(-1), marker="*", s=130, label="First BO observation")
ax.set(xlabel="x", ylabel="objective", title="First BO decision")
ax.legend(fontsize=8)
plt.show()

## 4. Commit the completed observation

candidateを選んだだけではtraining dataは更新しません。高価な評価が完了して`Y`が得られた時点でcompleted observationとして追加します。

In [ ]:
train_X = torch.cat([train_X, candidate], dim=0)
train_Y = torch.cat([train_Y, observed_Y], dim=0)

assert train_X.shape[0] == N_INITIAL + 1
print("completed evaluations:", train_X.shape[0])

### 観測が完了したらデータへ追加する

`train_X` と `train_Y` に新しい観測を追加します。**候補を選んだだけではデータを追加しません**。

次の反復では、更新済みデータからGPを新しく構築し、`fit()` で再学習します。

## 5. Budgeted sequential BO loop

ここから残りの評価予算を消費します。履歴にはcandidate、観測値、incumbentを保存し、BO campaign終了後に意思決定履歴を追跡できるようにします。

### 評価予算が尽きるまで繰り返す

各反復で、GP学習 → 獲得関数構築 → 候補提案 → 実験 → データ追加を実行します。

履歴には候補 `candidate`、観測値 `observed`、その時点までの最良観測入力 `incumbent_x` と最良値 `incumbent_y` を記録します。**incumbentは必ず観測済みデータから選びます**。

In [ ]:
history = []
for step in range(1, N_BO_STEPS + 1):
    model = SingleTaskGP(train_X, train_Y)
    model.fit()
    model.eval()

    acquisition = qLogExpectedImprovement(
        model=model,
        best_f=train_Y.max(),
        sampler=SobolQMCNormalSampler(
            torch.Size([128]),
            seed=200 + step,
        ),
    )
    candidate, acq_value = optimize_acqf(
        acq_function=acquisition,
        bounds=bounds,
        q=1,
        optimizer="botorch",
        num_restarts=4,
        raw_samples=64,
    )
    candidate = candidate.detach()
    observed_Y = expensive_objective(candidate)
    train_X = torch.cat([train_X, candidate], dim=0)
    train_Y = torch.cat([train_Y, observed_Y], dim=0)

    incumbent_index = train_Y.squeeze(-1).argmax()
    history.append(
        {
            "step": step,
            "candidate": float(candidate.item()),
            "observed": float(observed_Y.item()),
            "incumbent_x": float(train_X[incumbent_index].item()),
            "incumbent_y": float(train_Y[incumbent_index].item()),
            "acquisition": float(acq_value.item()),
        }
    )

for record in history:
    print(record)

### 最良観測値と予測最大値は異なる

`incumbent` は**実際に測定した候補のうち最良のもの**です。GP事後平均が最大となる入力は、まだ測定していない可能性があるため別の量です。

獲得関数値も改善の尺度であり、目的関数値として履歴に記録してはいけません。

## 6. Campaign result and stopping condition

この例の停止条件は固定評価予算です。実務では時間、コスト、改善量、uncertainty、feasibilityなどを停止条件にできますが、acquisition valueだけを普遍的な停止閾値とはみなしません。

In [ ]:
best_index = train_Y.squeeze(-1).argmax()
best_X = train_X[best_index]
best_Y = train_Y[best_index]
total_evaluations = train_X.shape[0]
expected_evaluations = N_INITIAL + 1 + N_BO_STEPS

assert total_evaluations == expected_evaluations
print("total completed evaluations:", total_evaluations)
print("final incumbent X:", best_X.item())
print("final incumbent Y:", best_Y.item())

### 反復ごとの改善を可視化する

履歴に記録した観測値とincumbentの推移を表示します。incumbentは最大化問題なので**単調非減少**です。個々の新しい観測値は改善しないこともあります。

初期最良値も基準線として表示し、BOによってどの程度改善したか確認します。

In [ ]:
initial_best = initial_Y.max().item()
steps = [record["step"] for record in history]
observed_values = [record["observed"] for record in history]
best_values = [record["incumbent_y"] for record in history]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, observed_values, "o--", label="Observed at each step")
ax.plot(steps, best_values, "o-", label="Best observed so far")
ax.axhline(initial_best, linestyle=":", color="gray", label="Initial best")
ax.set(xlabel="Additional BO step", ylabel="Objective", title="Sequential BO progress")
ax.legend()
plt.show()
assert all(b >= a - 1e-10 for a, b in zip(best_values, best_values[1:]))

### 最終データでGPを再学習して予測を確認する

最終反復で追加した観測値は、ループ内で最後に学習したGPにはまだ反映されていません。そこで**全観測データから最終GPを再学習**して、最初のGPと予測を比較します。

In [ ]:
final_model = SingleTaskGP(train_X, train_Y)
final_model.fit()
final_model.eval()
with torch.no_grad():
    final_posterior = final_model.posterior(grid)
    final_mean = final_posterior.mean.squeeze(-1)
    final_std = final_posterior.variance.clamp_min(0).sqrt().squeeze(-1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(grid.squeeze(-1), expensive_objective(grid).squeeze(-1), label="True objective")
ax.plot(grid.squeeze(-1), initial_mean, label="Initial GP mean")
ax.plot(grid.squeeze(-1), final_mean, label="Final GP mean")
ax.fill_between(grid.squeeze(-1), final_mean - 2*final_std, final_mean + 2*final_std, alpha=0.2, label="Final ± 2 SD")
ax.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), s=22, label="All observations")
ax.scatter(best_X.squeeze(-1), best_Y.squeeze(-1), marker="*", s=140, label="Final incumbent")
ax.set(xlabel="x", ylabel="objective", title="GP after evaluation budget exhausted")
ax.legend(fontsize=8)
plt.show()

## 7. What belongs to campaign state?

このNotebookでは最小限としてcompleted `train_X/train_Y`と履歴を保持しました。実運用では次も明示的に保存する価値があります。

- objective direction and transforms
- search-space definition and candidate constraints
- random seed / model and acquisition configuration
- completed, pending, failed, and cancelled evaluations
- cost / fidelity / task metadata
- incumbent definition
- stopping reason
- model/version provenance

これらはGP state_dictだけでは復元できないBO campaignの状態です。

### 実運用で保存する状態

`train_X/train_Y` は完了した観測だけを保持します。実務では、未完了・失敗・キャンセルした試験、探索空間、目的の最大化/最小化、乱数シード、評価予算、使用モデルのバージョンも管理対象です。

Notebook 40の非同期BOでは未完了点を `X_pending` に保持します。**実験の進捗状態とGP学習データは別に管理**することが重要です。

## 8. Cross-layer responsibility map

| Layer | This workflow |
| --- | --- |
| Data | completed observations |
| Surrogate | `SingleTaskGP` |
| Fit | `SingleTaskGP.fit()` |
| Posterior | model uncertainty over candidate space |
| Sampling | Sobol QMC for MC acquisition |
| Acquisition | qLogEI |
| Initialization | optimizer-owned raw samples / restarts |
| Optimization | public `robotorchan.optim.optimize_acqf` |
| Candidate | nominal next experiment |
| Evaluation | external expensive objective |
| Update | append only completed observation |
| Stop | explicit evaluation budget |

## 9. Choose the workflow that matches the problem

このsequential workflowを全問題へ無理に拡張しません。

| Problem structure | Representative executable notebook |
| --- | --- |
| minimum SingleTaskGP BO | `01_single_task_gp.ipynb` |
| mixed continuous/categorical | `02_mixed_single_task_gp.ipynb` |
| multi-fidelity / cost-aware | `03_multi_fidelity_gp.ipynb` |
| MultiTask / Kronecker | `04_multitask_gp.ipynb` |
| high-dimensional reduction | `19_reduced_gp.ipynb` |
| robust observation surrogate | `10_robust_gp.ipynb` |
| multi-objective acquisition | `33_multiobjective_acquisition.ipynb` |
| regression active learning | `34_active_learning_acquisition.ipynb` |
| batch / async / fantasy | `40_batch_async_fantasization.ipynb` |
| TuRBO / trust region | `47_turbo_trust_region.ipynb` |
| execution input perturbation | `48_robust_input_perturbation.ipynb` |

## 10. Production boundary

Notebookはworkflow semanticsを示すExecutable Documentationです。実験キュー、永続DB、retry policy、worker scheduling、認証、監査ログなどのproduction orchestrationをNotebook helperへ隠しません。

同様に、モデル・acquisition・optimizerの選択を巨大な`run_bo()` helperへまとめないことで、各cross-layer contractを読み取れる状態に保ちます。

## 11. Related documentation

- `docs/theory/01_bayesian_optimization.md`
- `docs-internal/architecture/notebook-executable-documentation-architecture.md`
- `docs-internal/contracts/optimization/acquisition-integration.md`
- `docs-internal/maintenance/notebook-authoring-standard.md`

このNotebookはsequential single-objective BO campaignの基準点であり、各専門workflowは同じlifecycleの必要な層だけを置き換えます。

## 12. 実務に向けたチェックポイント

| 確認項目 | このNotebookでの対応 |
|---|---|
| 最大化する目的 | `expensive_objective()` |
| 初期データ | `N_INITIAL` 点の観測 |
| GP学習 | `SingleTaskGP.fit()` |
| 改善の獲得関数 | `qLogExpectedImprovement` |
| 次の候補 | `robotorchan.optim.optimize_acqf(q=1)` |
| 評価の完了 | `observed_Y` の取得後にX・Yを追加 |
| 観測最良値 | `train_Y.argmax()` で決定 |
| 終了条件 | 初期観測＋BO評価の固定予算 |
| 最終GP | 全観測を使って再学習 |

**重要：** 予測最良値、獲得関数値、実測の最良値は異なる量です。評価予算と完了済み観測の数を一貫して管理してください。